# Step 1: Frozen Encoder Baseline (Text + Image + Tabular Concat Fusion)

사전 계산된 Text(MiniLM), Image(CLIP), Tabular(SVD) feature bank를 불러와 각 modality tower로 64D projection을 만든 뒤, 세 vector를 concat하여 최종 64D game embedding을 생성합니다.

구성:
- Text: MiniLM 384D -> TextTower -> 64D
- Image: CLIP 512D -> ImageTower -> 64D
- Tabular: SVD 64D -> TabularTower -> 64D
- Fusion: Concat 192D -> FusionTower -> 64D

출력:
- `game_fusion/emb_game_concat_64.npy`
- `game_fusion/emb_game_concat_64.csv`


## 1. Import 및 기본 설정

In [ ]:
import importlib.util
import sys
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
if ROOT.name == "game_fusion":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 256
SEED = 42


def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)


seed_everything(SEED)

print(f"Root path: {ROOT}")
print(f"PyTorch device: {DEVICE}")


## 2. Game Catalog 및 ID 정렬 확인

Text/Tabular/Image bank를 game catalog의 `app_id` 순서에 맞춰 로드합니다. Image bank에는 일부 누락 game이 있을 수 있으므로 `load_image_bank(..., fill_missing=True)`에서 평균 vector로 채웁니다.

In [2]:
games = pd.read_parquet(ROOT / "Data_process" / "games_metadata_enriched.parquet")
game_ids = games["app_id"].to_numpy()

text_ids = pd.read_csv(ROOT / "text_data" / "emb_text_minilm.csv")["app_id"].to_numpy()
tab_ids = pd.read_csv(ROOT / "tabular_embedding" / "emb_tabular_svd64.csv")["app_id"].to_numpy()
image_ids = pd.read_csv(ROOT / "image_embedding" / "emb_clip_squash.csv")["app_id"].to_numpy()

print(f"Game catalog: {games.shape}")
print(f"Game IDs:    {game_ids.shape}")
print(f"Text IDs:    {text_ids.shape} | aligned={np.array_equal(game_ids, text_ids)}")
print(f"Tabular IDs: {tab_ids.shape} | aligned={np.array_equal(game_ids, tab_ids)}")
print(f"Image IDs:   {image_ids.shape} | exact_aligned={np.array_equal(game_ids, image_ids)}")
print(f"Image coverage in catalog: {np.isin(game_ids, image_ids).sum():,}/{len(game_ids):,}")


Game catalog: (50872, 42)
Game IDs:    (50872,)
Text IDs:    (50872,) | aligned=True
Tabular IDs: (50872,) | aligned=True
Image IDs:   (50864,) | exact_aligned=False
Image coverage in catalog: 50,864/50,872


## 3. Tower Class 및 Feature Bank 로드

In [3]:
from tabular_embedding.tabular_tower import TabularTower, load_tabular_bank
from game_fusion.fusion_tower import FusionTower

text_tower_path = ROOT / "text_data" / "08_text_tower.py"
spec = importlib.util.spec_from_file_location("text_tower", text_tower_path)
text_tower_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(text_tower_module)
TextTower = text_tower_module.TextTower
load_text_bank = text_tower_module.load_text_bank

image_tower_path = ROOT / "image_embedding" / "07_image_tower.py"
spec = importlib.util.spec_from_file_location("image_tower", image_tower_path)
image_tower_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(image_tower_module)
ImageTower = image_tower_module.ImageTower
load_image_bank = image_tower_module.load_image_bank

text_bank, text_id2row = load_text_bank(
    ROOT / "text_data" / "emb_text_minilm",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
image_bank, image_id2row = load_image_bank(
    ROOT / "image_embedding" / "emb_clip_squash",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)
tab_bank, tab_id2row = load_tabular_bank(
    ROOT / "tabular_embedding" / "emb_tabular_svd64",
    app_ids=game_ids,
    device=DEVICE,
    fill_missing=True,
)

print("Feature banks loaded")
print(f"  Text:    {tuple(text_bank.shape)} {text_bank.dtype}")
print(f"  Image:   {tuple(image_bank.shape)} {image_bank.dtype}")
print(f"  Tabular: {tuple(tab_bank.shape)} {tab_bank.dtype}")


[image_tower] 이미지 없는 8건을 평균 벡터로 채움: [np.int64(2381590), np.int64(661700), np.int64(451330), np.int64(1204870), np.int64(1398280), np.int64(1468960), np.int64(1611460), np.int64(2226640)]
Feature banks loaded
  Text:    (50872, 384) torch.float32
  Image:   (50872, 512) torch.float32
  Tabular: (50872, 64) torch.float32


## 4. Tower 초기화

각 modality는 64D로 맞춘 뒤 concat합니다. 세 modality를 사용하므로 FusionTower 입력은 192D입니다.

In [4]:
text_tower = TextTower(in_dim=384, hidden=192, out_dim=64).to(DEVICE)
image_tower = ImageTower(in_dim=512, hidden=256, out_dim=64).to(DEVICE)
tab_tower = TabularTower(in_dim=64, hidden=128, out_dim=64).to(DEVICE)
fusion_tower = FusionTower(in_dim=192, hidden=384, out_dim=64).to(DEVICE)

for name, tower in [
    ("TextTower", text_tower),
    ("ImageTower", image_tower),
    ("TabularTower", tab_tower),
    ("FusionTower", fusion_tower),
]:
    params = sum(p.numel() for p in tower.parameters() if p.requires_grad)
    print(f"{name}: trainable params={params:,}")


TextTower: trainable params=86,272
ImageTower: trainable params=147,776
TabularTower: trainable params=16,832
FusionTower: trainable params=99,520


## 5. Forward Pass 예시

In [5]:
batch_size = 32
batch_indices = np.random.randint(0, len(game_ids), batch_size)

z_text_batch = text_bank[batch_indices]
z_image_batch = image_bank[batch_indices]
z_tab_batch = tab_bank[batch_indices]

h_text = text_tower(z_text_batch)
h_image = image_tower(z_image_batch)
h_tab = tab_tower(z_tab_batch)
z_concat = torch.cat([h_text, h_image, h_tab], dim=-1)
game_embedding = fusion_tower(z_concat)

print(f"Text projection:    {h_text.shape}, norm={h_text.norm(dim=-1).mean():.4f}")
print(f"Image projection:   {h_image.shape}, norm={h_image.norm(dim=-1).mean():.4f}")
print(f"Tabular projection: {h_tab.shape}, norm={h_tab.norm(dim=-1).mean():.4f}")
print(f"Concat:             {z_concat.shape}")
print(f"Game embedding:     {game_embedding.shape}, norm={game_embedding.norm(dim=-1).mean():.4f}")


Text projection:    torch.Size([32, 64]), norm=1.0000
Image projection:   torch.Size([32, 64]), norm=1.0000
Tabular projection: torch.Size([32, 64]), norm=1.0000
Concat:             torch.Size([32, 192])
Game embedding:     torch.Size([32, 64]), norm=1.0000


## 6. 전체 Game Embedding 생성

In [6]:
text_tower.eval()
image_tower.eval()
tab_tower.eval()
fusion_tower.eval()

n_games = len(game_ids)
all_embeddings = []

print(f"Generating {n_games:,} game embeddings with Text+Image+Tabular fusion...")

with torch.no_grad():
    for start_idx in range(0, n_games, BATCH_SIZE):
        end_idx = min(start_idx + BATCH_SIZE, n_games)

        z_text = text_bank[start_idx:end_idx]
        z_image = image_bank[start_idx:end_idx]
        z_tab = tab_bank[start_idx:end_idx]

        h_text = text_tower(z_text)
        h_image = image_tower(z_image)
        h_tab = tab_tower(z_tab)

        z_concat = torch.cat([h_text, h_image, h_tab], dim=-1)
        game_emb = fusion_tower(z_concat)
        all_embeddings.append(game_emb.cpu().numpy())

        if (start_idx // BATCH_SIZE + 1) % 20 == 0:
            print(f"  Processed {end_idx:,}/{n_games:,}")

game_embeddings = np.concatenate(all_embeddings, axis=0).astype(np.float32)
norms = np.linalg.norm(game_embeddings, axis=1)

print(f"Generated embeddings: {game_embeddings.shape}, dtype={game_embeddings.dtype}")
print(f"Norm min/mean/max: {norms.min():.6f} / {norms.mean():.6f} / {norms.max():.6f}")


Generating 50,872 game embeddings with Text+Image+Tabular fusion...
  Processed 5,120/50,872
  Processed 10,240/50,872
  Processed 15,360/50,872
  Processed 20,480/50,872
  Processed 25,600/50,872
  Processed 30,720/50,872
  Processed 35,840/50,872
  Processed 40,960/50,872
  Processed 46,080/50,872
Generated embeddings: (50872, 64), dtype=float32
Norm min/mean/max: 1.000000 / 1.000000 / 1.000000


## 7. 결과 저장

Step 2와 Step 3이 그대로 이어지도록 기존 파일명 `emb_game_concat_64.npy/csv`를 갱신합니다.

In [7]:
output_dir = ROOT / "game_fusion"
output_dir.mkdir(parents=True, exist_ok=True)

emb_npy_path = output_dir / "emb_game_concat_64.npy"
emb_csv_path = output_dir / "emb_game_concat_64.csv"

np.save(emb_npy_path, game_embeddings)
pd.DataFrame({"app_id": game_ids}).to_csv(emb_csv_path, index=False)

print(f"Saved embeddings: {emb_npy_path}")
print(f"Saved app_id map: {emb_csv_path}")
print(f"Shape: {game_embeddings.shape}")
print(f"Size: {game_embeddings.nbytes / 1024 / 1024:.2f} MB")


Saved embeddings: c:\Users\User\26_2_Contest\game_fusion\emb_game_concat_64.npy
Saved app_id map: c:\Users\User\26_2_Contest\game_fusion\emb_game_concat_64.csv
Shape: (50872, 64)
Size: 12.42 MB


## 8. Parameter Summary

In [8]:
text_params = sum(p.numel() for p in text_tower.parameters() if p.requires_grad)
image_params = sum(p.numel() for p in image_tower.parameters() if p.requires_grad)
tab_params = sum(p.numel() for p in tab_tower.parameters() if p.requires_grad)
fusion_params = sum(p.numel() for p in fusion_tower.parameters() if p.requires_grad)
total_params = text_params + image_params + tab_params + fusion_params

print("FROZEN ENCODER BASELINE - PARAMETER SUMMARY")
print(f"TextTower:    {text_params:,}")
print(f"ImageTower:   {image_params:,}")
print(f"TabularTower: {tab_params:,}")
print(f"FusionTower:  {fusion_params:,}")
print(f"Total:        {total_params:,}")
print(f"Step 1 complete: {game_embeddings.shape}")


FROZEN ENCODER BASELINE - PARAMETER SUMMARY
TextTower:    86,272
ImageTower:   147,776
TabularTower: 16,832
FusionTower:  99,520
Total:        350,400
Step 1 complete: (50872, 64)
